# MAE / MAPE / NRMSE / Pessimism — visual summary

Loads outputs of `compute_mae_mape.py`:
- `rebuttal_metrics_per_cell.csv` (per-cell metrics)
- `rebuttal_metrics_geomean.csv` (bucket-level geomean)
- `pessimism_overall.json` (per-(PDK, model) pessimism summary)

Bucket scheme is identical to `compare_topology.ipynb`:
(PDK × experiment × data_type × mode), with 4 models — `AADAM` / `MLP_MAML` / `GCN_Baseline` / `GCN_MAML`.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display, Markdown

HERE = os.path.dirname(os.path.abspath('view_mae_mape.ipynb')) if '__file__' not in dir() else os.path.dirname(os.path.abspath(__file__))
if not os.path.isdir(HERE):
    HERE = '/home/tkdgn2907/Deepsets_test/MAML/Projects/result_management/iccad2026_rebuttal'

per_cell  = pd.read_csv(os.path.join(HERE, 'rebuttal_metrics_per_cell.csv'))
geomean   = pd.read_csv(os.path.join(HERE, 'rebuttal_metrics_geomean.csv'))
pessimism = json.load(open(os.path.join(HERE, 'pessimism_overall.json')))

print(f'per_cell rows : {len(per_cell):>6}')
print(f'geomean rows  : {len(geomean):>6}')
print(f'source_labels : {sorted(per_cell["source_label"].unique())}')

In [ ]:
# === Style config — mirrors compare_topology.ipynb ===
MODEL_ORDER  = ['AADAM', 'MLP_MAML', 'GCN_Baseline', 'GCN_MAML']
MODEL_COLORS = {
    'AADAM':        '#b4d4b4',
    'MLP_MAML':     '#228b22',
    'GCN_Baseline': '#A8C4E0',
    'GCN_MAML':     '#1B5E91',
}
MODEL_DISPLAY = {
    'AADAM':        'MLP w/o MAML',
    'MLP_MAML':     'MLP_MAML',
    'GCN_Baseline': 'GCN w/o MAML',
    'GCN_MAML':     'GCN_MAML',
}
RATIO_PAIRS   = [(0, 1), (2, 3)]   # (baseline, MAML) — for ratio annotations
EXPERIMENTS   = [('topology_agnostic', 'Cross-Topology'),
                 ('intra_topology',    'Intra-Topology')]
GROUPS        = [('cell',       'interpolation', 'Cell-Inter'),
                 ('cell',       'extrapolation', 'Cell-Extra'),
                 ('transition', 'interpolation', 'Trans-Inter'),
                 ('transition', 'extrapolation', 'Trans-Extra')]
PDKS          = [('TSMC', 'Commercial'), ('ASAP7', 'ASAP7')]

plt.style.use('ggplot')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

## 1. Headline tables — geomean per metric

Pivot of `rebuttal_metrics_geomean.csv` so each row is one bucket (PDK × experiment × data_type × mode) and each column is a model.

In [ ]:
def pivot_metric(metric_col, value='geomean'):
    """Wide pivot: rows = (pdk, experiment, data_type, mode), cols = source_label."""
    col = f'{metric_col}_{value}'
    tbl = (geomean
        .pivot_table(index=['pdk','experiment','data_type','mode'],
                     columns='source_label', values=col, aggfunc='first')
        [MODEL_ORDER])
    tbl.columns = [MODEL_DISPLAY[c] for c in tbl.columns]
    return tbl

def fmt(tbl, n=3):
    return (tbl.style
        .format(f'{{:,.{n}f}}')
        .background_gradient(cmap='RdYlGn_r', axis=1)
        .set_properties(**{'text-align':'right'}))

for metric, label, ndig in [
    ('NRMSE',     'NRMSE (%) — geomean',         3),
    ('MAPE_pct',  'MAPE (%) — geomean',          3),
    ('MAE_scaled','MAE (ps for TSMC ns→ps, raw scale otherwise) — geomean', 4),
    ('MAE',       'MAE (raw units) — geomean',   5),
]:
    display(Markdown(f'### {label}'))
    display(fmt(pivot_metric(metric), ndig))

## 2. MAML vs Baseline ratio — "how much does MAML help?"

MLP ratio = AADAM / MLP_MAML, GCN ratio = GCN_Baseline / GCN_MAML. > 1 means MAML wins.

In [ ]:
def ratio_table(metric_col, value='geomean'):
    t = pivot_metric(metric_col, value).copy()
    out = pd.DataFrame(index=t.index)
    out['MLP ratio (no-MAML / MAML)'] = t['MLP w/o MAML'] / t['MLP_MAML']
    out['GCN ratio (no-MAML / MAML)'] = t['GCN w/o MAML'] / t['GCN_MAML']
    return out

for metric, label in [
    ('NRMSE',    'NRMSE ratio'),
    ('MAPE_pct', 'MAPE ratio'),
    ('MAE_scaled','MAE_scaled ratio'),
]:
    display(Markdown(f'### {label}'))
    display(
        ratio_table(metric).style
        .format('{:,.2f}x')
        .background_gradient(cmap='Greens', axis=None, vmin=1.0, vmax=10.0)
    )

## 3. Grouped bar plots — same layout as `compare_topology.ipynb`

Each chart: 2 sub-plots (TSMC / ASAP7), each showing 2 experiments × 4 buckets × 4 models. Red `xN.Nx` annotations mark MAML wins.

Identical visual style to the notebook figure — just swap which metric is on the y-axis.

In [ ]:
def lookup_geomean(pdk, model, experiment, data_type, mode, metric):
    col = f'{metric}_geomean'
    row = geomean[
        (geomean['pdk']==pdk) &
        (geomean['source_label']==model) &
        (geomean['experiment']==experiment) &
        (geomean['data_type']==data_type) &
        (geomean['mode']==mode)
    ]
    if len(row) == 0:
        return np.nan
    return float(row[col].iloc[0])


def plot_metric(metric, y_label, title_suffix='', log_y=False,
                fig_width=14.25, fig_height=3.8, bar_width=0.12,
                group_pad=0.08, section_gap=0.5, y_headroom=1.5):
    n_models     = len(MODEL_ORDER)
    n_groups     = len(GROUPS)
    n_experiments= len(EXPERIMENTS)
    group_width  = n_models * bar_width + group_pad
    section_width= n_groups * group_width

    fig, axes = plt.subplots(1, 2, figsize=(fig_width, fig_height))

    for ax_idx, (pdk, pdk_label) in enumerate(PDKS):
        ax = axes[ax_idx]
        max_val = 0
        section_centers = []
        positions = {}

        for e_idx, (experiment, exp_label) in enumerate(EXPERIMENTS):
            section_start = e_idx * (section_width + section_gap)
            section_centers.append(section_start + section_width / 2)

            for g_idx, (data_type, mode, _) in enumerate(GROUPS):
                for m_idx, model in enumerate(MODEL_ORDER):
                    val = lookup_geomean(pdk, model, experiment, data_type, mode, metric)
                    x = section_start + g_idx * group_width + m_idx * bar_width
                    positions[(e_idx, g_idx, m_idx)] = (x, val)
                    if not np.isnan(val):
                        ax.bar(x, val, width=bar_width * 0.85,
                               color=MODEL_COLORS[model], edgecolor='black', linewidth=0.8)
                        max_val = max(max_val, val)

        # MAML / Baseline ratio annotations
        for e_idx in range(n_experiments):
            for g_idx in range(n_groups):
                for base_idx, maml_idx in RATIO_PAIRS:
                    bk = (e_idx, g_idx, base_idx)
                    mk = (e_idx, g_idx, maml_idx)
                    if bk not in positions or mk not in positions:
                        continue
                    bx, bv = positions[bk]; mx, mv = positions[mk]
                    if np.isnan(bv) or np.isnan(mv) or mv <= 0:
                        continue
                    ratio = bv / mv
                    if ratio <= 1.05:
                        continue
                    ax.annotate('', xy=((bx+mx)/2, mv + 0.02*max_val),
                                xytext=((bx+mx)/2, bv - 0.02*max_val),
                                arrowprops=dict(arrowstyle='->', color='#C0392B', lw=1.3))
                    ax.text((bx+mx)/2, bv + 0.03*max_val, f'x{ratio:.1f}',
                            fontsize=8, fontweight='bold', color='#C0392B',
                            va='bottom', ha='center')

        # Section divider
        actual_bar_area_end = (n_groups - 1) * group_width + n_models * bar_width
        divider_x = (actual_bar_area_end + section_width + section_gap) / 2
        ax.axvline(x=divider_x, color='black', linestyle='--', linewidth=1.5, alpha=0.7)

        # X-tick labels per bucket
        centers, labels = [], []
        for e_idx in range(n_experiments):
            section_start = e_idx * (section_width + section_gap)
            for g_idx in range(n_groups):
                centers.append(section_start + g_idx * group_width + (n_models - 1) * bar_width / 2)
                labels.append(GROUPS[g_idx][2])
        ax.set_xticks(centers)
        ax.set_xticklabels(labels, fontsize=10, rotation=20, ha='right')

        # Y-axis
        if log_y:
            ax.set_yscale('log')
        else:
            ax.set_ylim(0, max_val * (1 + 0.25 if max_val > 0 else 1) + 0.1)
        if ax_idx == 0:
            ax.set_ylabel(y_label, fontsize=13, fontweight='bold')

        ax.set_title(pdk_label, fontsize=14, fontweight='bold', pad=22)
        for e_idx, (_, exp_label) in enumerate(EXPERIMENTS):
            ax.annotate(exp_label,
                xy=(section_centers[e_idx], (max_val if not log_y else max_val) * (1.18 if not log_y else 1.0)),
                xytext=(0, 6), textcoords='offset points',
                fontsize=12, fontweight='bold', ha='center', va='bottom')

    # Legend on the right of the figure
    handles = [Patch(facecolor=MODEL_COLORS[m], edgecolor='black', label=MODEL_DISPLAY[m])
               for m in MODEL_ORDER]
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 1.02),
               ncol=4, fontsize=11, frameon=False)
    fig.suptitle(f'{metric}{title_suffix}', fontsize=15, fontweight='bold', y=1.10)
    plt.tight_layout()
    plt.show()
    return fig

In [ ]:
plot_metric('NRMSE',    y_label='NRMSE (%)',  title_suffix=' (geomean)')

In [ ]:
plot_metric('MAPE_pct', y_label='MAPE (%)',  title_suffix=' (geomean)')

In [ ]:
# MAE_scaled uses heterogeneous units (ps for TSMC ns→ps, raw scale otherwise) → split by PDK
for pdk_only in ['TSMC', 'ASAP7']:
    fig, ax = plt.subplots(figsize=(8, 3.6))
    sub = (geomean[geomean['pdk']==pdk_only]
           .pivot_table(index=['experiment','data_type','mode'],
                        columns='source_label', values='MAE_scaled_geomean',
                        aggfunc='first')[MODEL_ORDER])
    sub.columns = [MODEL_DISPLAY[c] for c in sub.columns]
    sub.plot.bar(ax=ax, color=[MODEL_COLORS[m] for m in MODEL_ORDER],
                 edgecolor='black', linewidth=0.6, width=0.8)
    ax.set_title(f'{pdk_only} — MAE_scaled (geomean)', fontweight='bold')
    ax.set_ylabel('MAE_scaled')
    ax.tick_params(axis='x', rotation=35)
    ax.legend(fontsize=8, ncol=2, loc='upper left')
    plt.tight_layout(); plt.show()

## 4. Pessimism summary (P0-4 / A-Q2)

From `pessimism_overall.json`: per (PDK, model) — fraction of under-prediction (unsafe), p95 of safe-side pessimism, worst-case under-prediction.

In [ ]:
pess_df = (pd.DataFrame(pessimism).T
    .reset_index().rename(columns={'index':'key'}))
pess_df[['pdk','source_label']] = pess_df['key'].str.split('/', n=1, expand=True)
pess_df = pess_df.drop(columns='key').set_index(['pdk','source_label']).loc[
    pd.MultiIndex.from_product([['TSMC','ASAP7'], MODEL_ORDER]), :]

display(Markdown('### Per-PDK / per-model pessimism'))
display(pess_df.style
    .format({'n_cells':'{:,.0f}',
             'mean_underpred_frac':'{:.3f}',
             'max_underpred_frac':'{:.3f}',
             'mean_pess_p95':'{:.4f}',
             'mean_max_underpred':'{:.3f}',
             'max_max_underpred':'{:.3f}'})
    .background_gradient(cmap='RdYlGn_r', subset=['mean_underpred_frac','max_underpred_frac'])
    .background_gradient(cmap='RdYlGn_r', subset=['mean_max_underpred','max_max_underpred']))

In [ ]:
# Side-by-side: under-prediction fraction (want ~0.5 ish) and max under-prediction (want low).
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (col, ylab, title) in zip(axes, [
    ('mean_underpred_frac', 'mean P(pred < act)',  'Under-prediction fraction'),
    ('mean_max_underpred',  'mean max underpred',  'Worst-case underprediction (scaled)'),
]):
    w = 0.18
    for i, model in enumerate(MODEL_ORDER):
        xs, ys = [], []
        for j, (pdk, _) in enumerate(PDKS):
            ys.append(pess_df.loc[(pdk, model), col])
            xs.append(j + (i - 1.5) * w)
        ax.bar(xs, ys, width=w*0.95,
               color=MODEL_COLORS[model], edgecolor='black', linewidth=0.6,
               label=MODEL_DISPLAY[model])
    ax.set_xticks(range(len(PDKS)))
    ax.set_xticklabels([p[0] for p in PDKS], fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(ylab)
    ax.legend(fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

## 5. Per-cell distribution — does MAML help every cell or just on average?

For each (PDK, source_label), show distribution of `MAPE_pct` across cells. Spread means uneven performance; tight + low means consistently good.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
for ax, pdk in zip(axes, ['TSMC', 'ASAP7']):
    data, labels, colors = [], [], []
    for m in MODEL_ORDER:
        s = per_cell[(per_cell['pdk']==pdk) & (per_cell['source_label']==m)]['MAPE_pct'].dropna()
        if len(s) == 0:
            continue
        data.append(s.values)
        labels.append(MODEL_DISPLAY[m])
        colors.append(MODEL_COLORS[m])
    bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True,
                    meanprops=dict(marker='D', markerfacecolor='white',
                                   markeredgecolor='black', markersize=5))
    for patch, c in zip(bp['boxes'], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.8)
    ax.set_title(f'{pdk} — per-cell MAPE distribution', fontweight='bold')
    ax.set_ylabel('MAPE (%)')
    ax.set_yscale('log')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 6. Under-estimate vs Over-estimate ratio

For each prediction, *under-estimate* means the model predicted **smaller** than the actual value (`pred < act`), which is the **unsafe** direction for delay (it would let a fast path slip through STA). *Over-estimate* (`pred > act`) is the safe direction.

- `UnderPred_frac` column (per cell) = `mean(act > pred)`.
- We average this fraction across cells for each (PDK × source_label) bucket — and also break it down per (experiment × data_type × mode).

In [ ]:
# Per (PDK × source_label): mean under-fraction across cells (= mean per-cell P(act > pred)).
under_overall = (per_cell.dropna(subset=['UnderPred_frac'])
    .groupby(['pdk','source_label'])
    .agg(n_cells=('UnderPred_frac','size'),
         under_frac=('UnderPred_frac','mean'))
    .reset_index())
under_overall['over_frac'] = 1.0 - under_overall['under_frac']

# Pivot to (PDK, source_label) — display in canonical MODEL_ORDER, columns = under/over %.
under_wide = (under_overall
    .set_index(['pdk','source_label'])
    .loc[pd.MultiIndex.from_product([['TSMC','ASAP7'], MODEL_ORDER])])

display(Markdown('### Under-estimate vs Over-estimate fraction — per (PDK, model)'))
display(under_wide.style
    .format({'n_cells':'{:.0f}', 'under_frac':'{:.1%}', 'over_frac':'{:.1%}'})
    .background_gradient(cmap='RdYlGn_r', subset=['under_frac'], vmin=0.4, vmax=0.6))

In [ ]:
# Stacked bar — under (red) vs over (green), one bar per (PDK, model). Reference 50% line.
fig, ax = plt.subplots(figsize=(11, 3.8))
x_labels, x_pos = [], []
xi = 0
for pdk, _ in PDKS:
    for m in MODEL_ORDER:
        row = under_wide.loc[(pdk, m)]
        u, o = float(row['under_frac']), float(row['over_frac'])
        ax.bar(xi, u, color='#D9534F', edgecolor='black', linewidth=0.6, label='Under (pred < act)' if xi == 0 else None)
        ax.bar(xi, o, bottom=u, color='#5CB85C', edgecolor='black', linewidth=0.6, label='Over (pred > act)' if xi == 0 else None)
        ax.text(xi, u/2,         f'{u:.0%}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        ax.text(xi, u + o/2,     f'{o:.0%}', ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        x_labels.append(f'{pdk}\n{MODEL_DISPLAY[m]}')
        x_pos.append(xi)
        xi += 1
    xi += 0.6  # gap between PDKs

ax.axhline(0.5, color='black', linestyle='--', linewidth=1.2, alpha=0.7, label='50% (unbiased)')
ax.set_xticks(x_pos)
ax.set_xticklabels(x_labels, fontsize=9)
ax.set_ylabel('Fraction of predictions', fontweight='bold')
ax.set_ylim(0, 1)
ax.set_title('Under-estimate (unsafe) vs Over-estimate (safe) — averaged across cells',
             fontweight='bold')
ax.legend(loc='upper right', fontsize=9, ncol=3)
plt.tight_layout(); plt.show()

In [ ]:
# Per-bucket heatmap of under-estimate fraction (one heatmap per PDK).
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for ax, (pdk, pdk_label) in zip(axes, PDKS):
    sub = (per_cell[per_cell['pdk']==pdk]
        .dropna(subset=['UnderPred_frac'])
        .groupby(['source_label','experiment','data_type','mode'])
        ['UnderPred_frac'].mean()
        .reset_index())
    sub['bucket'] = sub['experiment'].map({'topology_agnostic':'X-Topo',
                                            'intra_topology':   'Intra'}) + ' | ' + \
                    sub['data_type'].map({'cell':'Cell',
                                          'transition':'Trans'}) + '-' + \
                    sub['mode'].map({'interpolation':'Inter',
                                     'extrapolation':'Extra'})
    bucket_order = [f'{e} | {d}-{m}'
                    for e in ['X-Topo','Intra']
                    for d in ['Cell','Trans']
                    for m in ['Inter','Extra']]
    pivot = (sub.pivot(index='source_label', columns='bucket', values='UnderPred_frac')
        .reindex(index=MODEL_ORDER, columns=bucket_order))
    pivot.index = [MODEL_DISPLAY[m] for m in pivot.index]

    im = ax.imshow(pivot.values, cmap='RdYlGn_r', vmin=0.35, vmax=0.65, aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha='right', fontsize=9)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=10)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.0%}', ha='center', va='center',
                        fontsize=8.5, fontweight='bold',
                        color='white' if abs(v-0.5) > 0.08 else 'black')
    ax.set_title(f'{pdk_label} — Under-estimate fraction per bucket', fontweight='bold')
    fig.colorbar(im, ax=ax, shrink=0.85, label='under-frac (0.5 = unbiased)')

plt.tight_layout(); plt.show()

## 7. Quick numeric digest — for the rebuttal text

One-paragraph friendly summary auto-generated from the geomean table — useful for citing concrete numbers in the rebuttal.

In [ ]:
def gm(pdk, model, exp, dt, mode, metric):
    return lookup_geomean(pdk, model, exp, dt, mode, metric)

lines = []
for pdk, _ in PDKS:
    for exp_key, exp_pretty in [('topology_agnostic','Cross-topology'),
                                 ('intra_topology','Intra-topology')]:
        for dt, mode, dt_label in [
            ('cell','extrapolation','Cell-Extra'),
            ('transition','extrapolation','Trans-Extra'),
        ]:
            aadam   = gm(pdk,'AADAM',       exp_key, dt, mode, 'NRMSE')
            mlpmaml = gm(pdk,'MLP_MAML',    exp_key, dt, mode, 'NRMSE')
            gcnb    = gm(pdk,'GCN_Baseline',exp_key, dt, mode, 'NRMSE')
            gcnm    = gm(pdk,'GCN_MAML',    exp_key, dt, mode, 'NRMSE')
            mape_m  = gm(pdk,'GCN_MAML',    exp_key, dt, mode, 'MAPE_pct')
            lines.append(
                f'**{pdk}** / {exp_pretty} / {dt_label} — '
                f'NRMSE: AADAM={aadam:.2f} → MLP_MAML={mlpmaml:.2f} '
                f'(x{aadam/mlpmaml:.1f}); '
                f'GCN_Base={gcnb:.2f} → GCN_MAML={gcnm:.2f} '
                f'(x{gcnb/gcnm:.1f}); '
                f'GCN_MAML MAPE={mape_m:.2f}%.'
            )
display(Markdown('\n\n'.join(lines)))